### Byte-Pair Encoding

In this notebook, you'll implement the byte-pair encoding algorithm to build a tokenizer.

In [1]:
import glob
import re

from collections import Counter

We'll build this on a small piece of text. Once you get the code working, you can try it out on a larger corpus and/or try more iterations.

In [2]:
text = """
Did you ever hear the tragedy of Darth Plagueis The Wise? I thought not. It’s not a story the Jedi would tell you. It’s a Sith legend. Darth Plagueis was a Dark Lord of the Sith, so powerful and so wise he could use the Force to influence the midichlorians to create life… He had such a knowledge of the dark side that he could even keep the ones he cared about from dying. The dark side of the Force is a pathway to many abilities some consider to be unnatural. He became so powerful… the only thing he was afraid of was losing his power, which eventually, of course, he did. Unfortunately, he taught his apprentice everything he knew, then his apprentice killed him in his sleep. Ironic. He could save others from death, but not himself.
"""

print(text)


Did you ever hear the tragedy of Darth Plagueis The Wise? I thought not. It’s not a story the Jedi would tell you. It’s a Sith legend. Darth Plagueis was a Dark Lord of the Sith, so powerful and so wise he could use the Force to influence the midichlorians to create life… He had such a knowledge of the dark side that he could even keep the ones he cared about from dying. The dark side of the Force is a pathway to many abilities some consider to be unnatural. He became so powerful… the only thing he was afraid of was losing his power, which eventually, of course, he did. Unfortunately, he taught his apprentice everything he knew, then his apprentice killed him in his sleep. Ironic. He could save others from death, but not himself.



We'll start by preprocessing the text, lowercasing all text and splitting on spaces.

In [3]:
words = text.lower().split()
print(f"Total Words: {len(words)}")

merge_rules = []

Total Words: 138


**Part 1:** Create a function, tokenize_word that takes as input as word and outputs that word converted to a tuple of characters plus an end of word character, `</w>`.

**Example Input**: `'darth'`  
**Example Output:** `('d', 'a', 'r', 't', 'h', '</w>')`.

In [4]:
def tokenize_word(word:str) -> tuple[str]:
    return tuple(word) + ('</w>',)

Now, we can apply this function across each word.

In [5]:
tokenized = [tokenize_word(word) for word in words]
tokenized[:5]

[('d', 'i', 'd', '</w>'),
 ('y', 'o', 'u', '</w>'),
 ('e', 'v', 'e', 'r', '</w>'),
 ('h', 'e', 'a', 'r', '</w>'),
 ('t', 'h', 'e', '</w>')]

We'll take the result of this and build a vocabulary counter.

In [6]:
vocab = Counter(tokenized)
vocab

Counter({('t', 'h', 'e', '</w>'): 11,
         ('h', 'e', '</w>'): 10,
         ('o', 'f', '</w>'): 6,
         ('a', '</w>'): 5,
         ('t', 'o', '</w>'): 4,
         ('h', 'i', 's', '</w>'): 4,
         ('w', 'a', 's', '</w>'): 3,
         ('d', 'a', 'r', 'k', '</w>'): 3,
         ('s', 'o', '</w>'): 3,
         ('c', 'o', 'u', 'l', 'd', '</w>'): 3,
         ('d', 'a', 'r', 't', 'h', '</w>'): 2,
         ('p', 'l', 'a', 'g', 'u', 'e', 'i', 's', '</w>'): 2,
         ('i', 't', '’', 's', '</w>'): 2,
         ('n', 'o', 't', '</w>'): 2,
         ('f', 'o', 'r', 'c', 'e', '</w>'): 2,
         ('s', 'i', 'd', 'e', '</w>'): 2,
         ('f', 'r', 'o', 'm', '</w>'): 2,
         ('a', 'p', 'p', 'r', 'e', 'n', 't', 'i', 'c', 'e', '</w>'): 2,
         ('d', 'i', 'd', '</w>'): 1,
         ('y', 'o', 'u', '</w>'): 1,
         ('e', 'v', 'e', 'r', '</w>'): 1,
         ('h', 'e', 'a', 'r', '</w>'): 1,
         ('t', 'r', 'a', 'g', 'e', 'd', 'y', '</w>'): 1,
         ('w', 'i', 's', 'e', '?', '<

**Part 2:** Create a function, create_pairs, that takes as input a tuple of strings and which returns a list containing all tuples that can be created by taking pairs on consecutive elements of the input tuple.

**Example Input:** `('n', 'o', 't', '</w>')`  
**Example Output:** `[('n', 'o'), ('o', 't'), ('t', '</w>')]`

In [7]:
def create_pairs(tup:tuple[str]) -> list[tuple[str]]:
    new_tup_list = []
    current_tup = ()
    for i, x in enumerate(tup):
        if i == 0:
            current_tup = (x, x)
        else:
            current_tup = (current_tup[1], x)
            new_tup_list.append(current_tup)
    return new_tup_list

# Alternate
# def create_pairs(tup:tuple) -> list[tuple]:
#     return list(zip(tup, tup[1:]))

In [8]:
create_pairs(('n', 'o', 't', '</w>'))

[('n', 'o'), ('o', 't'), ('t', '</w>')]

Now, we'll use the function along with the vocabulary frequencies to count the most common pairs.

In [9]:
pairs = []
for word, freq in vocab.items():
    pairs.extend(create_pairs(word) * freq)

pairs = Counter(pairs)
pairs.most_common(1)

[(('e', '</w>'), 36)]

**Part 3:** Now, we need to merge the most common pair.

To do this, you may want to create a function that accepts a pair_to_merge and a word tuple and returns the word tuple after merging any instance of the pair_to_merge.

**Example Input:** word = `('t', 'h', 'e', '</w>')`, pair_to_merge =  `('e', '</w>')`  
**Example Output:** `('t', 'h', 'e</w>')`

**Example Input:** word = `('d', 'a', 'r', 't', 'h', '</w>')`, pair_to_merge =  `('e', '</w>')`  
**Example Output:** `('d', 'a', 'r', 't', 'h', '</w>')`

In [10]:
def merge_pair(word:tuple[str], pair_to_merge:tuple[str]) -> tuple[str]:
    word = list(word)
    
    for i, w in enumerate(word):
        if word[i-1] == pair_to_merge[0] and word[i] == pair_to_merge[1]:
            word[i-1] = word[i-1] + word[i]
            word.pop(i)

    return tuple(word)

# Alternate
# def merge_pair(word:tuple[str], pair_to_merge: tuple[str]) -> tuple[str]:
#     result = []
#     i = 0

#     while i < len(word):
#         if word[i:i+2] == pair_to_merge:
#             result.append("".join(pair_to_merge))
#             i = i + 2
#         else:
#             result.append(word[i])
#             i = i + 1

#     return tuple(result)

In [11]:
merge_pair(word = ('t', 'h', 'e', '</w>'), pair_to_merge = ('e', '</w>'))

('t', 'h', 'e</w>')

In [12]:
merge_pair(word = ('d', 'a', 'r', 't', 'h', '</w>'), pair_to_merge = ('e', '</w>'))

('d', 'a', 'r', 't', 'h', '</w>')

Now, we can apply this function across the vocabulary to create our new vocabulary.

In [13]:
new_vocab = {}
new_merge = pairs.most_common()[0][0]
merge_rules.append(new_merge)

for word in vocab:
    new_vocab[merge_pair(word, new_merge)] = vocab[word]

vocab = new_vocab

In [14]:
vocab

{('d', 'i', 'd', '</w>'): 1,
 ('y', 'o', 'u', '</w>'): 1,
 ('e', 'v', 'e', 'r', '</w>'): 1,
 ('h', 'e', 'a', 'r', '</w>'): 1,
 ('t', 'h', 'e</w>'): 11,
 ('t', 'r', 'a', 'g', 'e', 'd', 'y', '</w>'): 1,
 ('o', 'f', '</w>'): 6,
 ('d', 'a', 'r', 't', 'h', '</w>'): 2,
 ('p', 'l', 'a', 'g', 'u', 'e', 'i', 's', '</w>'): 2,
 ('w', 'i', 's', 'e', '?', '</w>'): 1,
 ('i', '</w>'): 1,
 ('t', 'h', 'o', 'u', 'g', 'h', 't', '</w>'): 1,
 ('n', 'o', 't', '.', '</w>'): 1,
 ('i', 't', '’', 's', '</w>'): 2,
 ('n', 'o', 't', '</w>'): 2,
 ('a', '</w>'): 5,
 ('s', 't', 'o', 'r', 'y', '</w>'): 1,
 ('j', 'e', 'd', 'i', '</w>'): 1,
 ('w', 'o', 'u', 'l', 'd', '</w>'): 1,
 ('t', 'e', 'l', 'l', '</w>'): 1,
 ('y', 'o', 'u', '.', '</w>'): 1,
 ('s', 'i', 't', 'h', '</w>'): 1,
 ('l', 'e', 'g', 'e', 'n', 'd', '.', '</w>'): 1,
 ('w', 'a', 's', '</w>'): 3,
 ('d', 'a', 'r', 'k', '</w>'): 3,
 ('l', 'o', 'r', 'd', '</w>'): 1,
 ('s', 'i', 't', 'h', ',', '</w>'): 1,
 ('s', 'o', '</w>'): 3,
 ('p', 'o', 'w', 'e', 'r', 'f', 'u',

**Part 4:** Write a for loop so that you can repeat the process of finding the most common pair and then updating the vocabulary. Run this for 10 iterations.

In [15]:
for _ in range(10):
    pairs = []
    for word, freq in vocab.items():
        pairs.extend(create_pairs(word) * freq)

    pairs = Counter(pairs)
    pairs.most_common(1)
    
    new_vocab = {}
    new_merge = pairs.most_common()[0][0]
    merge_rules.append(new_merge)

    for word in vocab:
        new_vocab[merge_pair(word, new_merge)] = vocab[word]

    vocab = new_vocab

Let's see the list of new tokens after the 11 iterations we have performed.

In [16]:
tokens = []
for word in vocab:
    tokens.extend(word)
tokens = list(set(tokens))
sorted(tokens)

[',',
 '.</w>',
 '</w>',
 '?',
 'a',
 'ar',
 'b',
 'c',
 'd',
 'd</w>',
 'e',
 'e</w>',
 'er',
 'f',
 'g',
 'h',
 'he</w>',
 'i',
 'is</w>',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'ou',
 'p',
 'r',
 's',
 's</w>',
 't',
 'th',
 'the</w>',
 'u',
 'v',
 'w',
 'y',
 '’',
 '…']

And here's our set of merge rules.

In [17]:
merge_rules

[('e', '</w>'),
 ('t', 'h'),
 ('s', '</w>'),
 ('d', '</w>'),
 ('th', 'e</w>'),
 ('h', 'e</w>'),
 ('o', 'u'),
 ('.', '</w>'),
 ('e', 'r'),
 ('a', 'r'),
 ('i', 's</w>')]

If we want to tokenize some new text, this is what it would look like. Note that we have to merge in the same order as we did with the original text.

In [18]:
new_text = "It's over Anakin, I have the high ground."
new_words = new_text.lower().split()

new_tokenized = [tokenize_word(word) for word in new_words]
new_tokenized

[('i', 't', "'", 's', '</w>'),
 ('o', 'v', 'e', 'r', '</w>'),
 ('a', 'n', 'a', 'k', 'i', 'n', ',', '</w>'),
 ('i', '</w>'),
 ('h', 'a', 'v', 'e', '</w>'),
 ('t', 'h', 'e', '</w>'),
 ('h', 'i', 'g', 'h', '</w>'),
 ('g', 'r', 'o', 'u', 'n', 'd', '.', '</w>')]

And then a helper function to apply byte-pair encoding using our rules.

In [19]:
def apply_bpe(word, merge_rules):
    word = list(word)
    while True:
        pairs = [(word[i], word[i+1]) for i in range(len(word)-1)]
        merge_found = False
        for merge in merge_rules:
            if merge in pairs:
                i = pairs.index(merge)
                word = word[:i] + [''.join(merge)] + word[i+2:]
                merge_found = True
                break
        if not merge_found:
            break
    return tuple(word)

In [20]:
[apply_bpe(word, merge_rules) for word in new_tokenized]

[('i', 't', "'", 's</w>'),
 ('o', 'v', 'er', '</w>'),
 ('a', 'n', 'a', 'k', 'i', 'n', ',', '</w>'),
 ('i', '</w>'),
 ('h', 'a', 'v', 'e</w>'),
 ('the</w>',),
 ('h', 'i', 'g', 'h', '</w>'),
 ('g', 'r', 'ou', 'n', 'd', '.</w>')]

**Bonus Challenge: Training a Larger Byte-Pair Encoding Tokenizer**

In the previous notebook, you worked with the top 50 Project Gutenberg books and created tokenized representations of each document. Instead of training byte-pair encoding on a single paragraph, let's see what happens when we train it on a much larger corpus.

Combine the text from all 50 Project Gutenberg books into a single large string and train your byte-pair encoding tokenizer on this larger corpus.

You may want to:
* reuse your import_book function from the previous notebook,
* iterate through all files using glob,
* concatenate the books together into one string.

In [21]:
def import_book(filepath:str) -> str:
    """
    Read book file and remove start and end metadata.
    """
    with open(file=filepath, encoding='utf-8') as fi:
        book = fi.read()

    start_metadata = re.search(
        pattern=r'\*\*\*.+\*\*\*',
        string=book
    )

    book = book[start_metadata.end():]

    end_metadata = re.search(
        pattern=r'\*\*\*.+',
        string=book
    )

    book = book[:end_metadata.start()]

    return book

In [22]:
books_filepath_list = glob.glob('books/*.txt')
len(books_filepath_list)

56

In [23]:
books_str = ''

for book_filepath in books_filepath_list:
    print(book_filepath)
    book = import_book(filepath=book_filepath)

    books_str = books_str + book

books\A Christmas Carol in Prose; Being a Ghost Story of Christmas by Charles Dickens.txt
books\A Modest Proposal by Jonathan Swift.txt
books\A Tale of Two Cities by Charles Dickens.txt
books\Adventures of Huckleberry Finn by Mark Twain.txt
books\Ang _Filibusterismo_.txt
books\Anna Karenina by graf Leo Tolstoy.txt
books\Anne of Green Gables by L. M.  Montgomery.txt
books\Anthem by Ayn Rand.txt
books\Crime and Punishment by Fyodor Dostoyevsky.txt
books\Don Quixote by Miguel de Cervantes Saavedra.txt
books\Dracula by Bram Stoker.txt
books\Dubliners by James Joyce.txt
books\Emma by Jane Austen.txt
books\Frankenstein; Or, The Modern Prometheus by Mary Wollstonecraft Shelley.txt
books\Great Expectations by Charles Dickens.txt
books\Heart of Darkness by Joseph Conrad.txt
books\Jane Eyre_ An Autobiography by Charlotte Brontë.txt
books\Japanese Girls and Women by Alice Mabel Bacon.txt
books\Les Misérables by Victor Hugo.txt
books\Leviathan by Thomas Hobbes.txt
books\Little Women by Louisa May 

In [35]:
words = books_str.lower().split()
print(f"Total Words: {len(words)}")

merge_rules = []

tokenized = [tokenize_word(word) for word in words]
print(tokenized[:5])

vocab = Counter(tokenized)
print(f"Vacab: {len(vocab)}")

Total Words: 6632180
[('a', '</w>'), ('c', 'h', 'r', 'i', 's', 't', 'm', 'a', 's', '</w>'), ('c', 'a', 'r', 'o', 'l', '</w>'), ('i', 'n', '</w>'), ('p', 'r', 'o', 's', 'e', '</w>')]
Vacab: 261000


In [36]:
for _ in range(10):
    pairs = []
    for word, freq in vocab.items():
        pairs.extend(create_pairs(word) * freq)

    pairs = Counter(pairs)
    pairs.most_common(1)
    
    new_vocab = {}
    new_merge = pairs.most_common()[0][0]
    merge_rules.append(new_merge)

    for word in vocab:
        new_vocab[merge_pair(word, new_merge)] = vocab[word]

    vocab = new_vocab

In [37]:
vocab_10 = vocab

merge_rules_10 = merge_rules
print(len(merge_rules_10))

10


In [38]:
for _ in range(40):
    pairs = []
    for word, freq in vocab.items():
        pairs.extend(create_pairs(word) * freq)

    pairs = Counter(pairs)
    pairs.most_common(1)
    
    new_vocab = {}
    new_merge = pairs.most_common()[0][0]
    merge_rules.append(new_merge)

    for word in vocab:
        new_vocab[merge_pair(word, new_merge)] = vocab[word]

    vocab = new_vocab

In [39]:
vocab_50 = vocab

merge_rules_50 = merge_rules
print(len(merge_rules_50))

50


Train your tokenizer using different numbers of merge operations such as:
* 10 merges
* 50 merges
* 100 merges

After training, investigate the learned tokens and answer the following questions:
* What kinds of subwords or patterns are learned?
* Do common suffixes or prefixes appear as tokens?
* Which tokens appear to represent full words?
* How does increasing the number of merges change the vocabulary?

In [40]:
len(vocab)

261000

In [41]:
len(vocab_50)

261000

In [42]:
merge_rules

[('e', '</w>'),
 ('t', 'h'),
 ('d', '</w>'),
 ('t', '</w>'),
 ('s', '</w>'),
 ('i', 'n'),
 ('a', 'n'),
 (',', '</w>'),
 ('e', 'r'),
 ('th', 'e</w>'),
 ('y', '</w>'),
 ('o', 'u'),
 ('o', 'n'),
 ('o', '</w>'),
 ('e', 'n'),
 ('.', '</w>'),
 ('an', 'd</w>'),
 ('o', 'r'),
 ('f', '</w>'),
 ('e', 'd</w>'),
 ('a', 'r'),
 ('in', 'g'),
 ('t', 'o</w>'),
 ('h', 'i'),
 ('o', 'f</w>'),
 ('r', 'e'),
 ('h', 'a'),
 ('l', 'l'),
 ('a', '</w>'),
 ('er', '</w>'),
 ('ing', '</w>'),
 ('s', 't'),
 ('a', 't</w>'),
 ('a', 's</w>'),
 ('in', '</w>'),
 ('h', 'e</w>'),
 ('o', 'm'),
 ('a', 't'),
 ('i', 't'),
 ('i', '</w>'),
 ('e', 's'),
 ('o', 'w'),
 ('c', 'h'),
 ('s', 'e'),
 ('w', 'i'),
 ('a', 'l'),
 ('on', '</w>'),
 ('or', '</w>'),
 ('”', '</w>'),
 ('ll', '</w>')]

**Additional Stretch Goal: Optimizing Byte-Pair Encoding**
The implementation of byte-pair encoding above recomputes pair frequencies across the entire vocabulary during every merge iteration.

This works well for small corpora, but can become slow when training on large collections of text such as the full Project Gutenberg corpus.

Modify your implementation so that it updates pair counts more efficiently.

Instead of recomputing all pairs after every merge:
* Keep track of which pairs occur inside each word.
* Create a reverse lookup that maps each pair to the words that contain it.
* After merging a pair, only update the words affected by that merge.

You may find the following data structures useful:

`word_to_pairs[word]`

Stores the list of consecutive pairs contained in a word.

**Example:**

`('t', 'h', 'e', '</w>')`

might map to:

`[('t', 'h'), ('h', 'e'), ('e', '</w>')]`

---

`pair_to_words[pair]`

Stores the set of words containing a particular pair.

Example:

`('t', 'h')`

might map to:

`{
    ('t', 'h', 'e', '</w>'),
    ('a', 'n', 't', 'h', 'e', 'm', '</w>')
}`


Then try running 1000 iterations or more on the full corpus and inspecting the results.